In [1]:
!pip3 install singlestoredb

DEPRECATION: Loading egg at /Users/pragalbh.devsingh/Documents/work/models/ColBERT/venv/lib/python3.12/site-packages/colbert_ai-0.2.20-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /Users/pragalbh.devsingh/Documents/work/models/ColBERT/venv/lib/python3.12/site-packages/ninja-1.11.1.3-py3.12-macosx-15.2-arm64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /Users/pragalbh.devsingh/Documents/work/models/ColBERT/venv/lib/python3.12/site-packages/transformers-4.49.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://githu

In [49]:
import singlestoredb as s2

conn=s2.connect(host='linkup-singlestore.prod.synaptic.services',
                     port=3306,password='ManagerMemSQL@synaptic_production',user='manager_user',
                     database='labs_core_reloaded_production',connect_timeout=30,results_type='dict')

In [50]:
cur=conn.cursor()

In [53]:

cur.execute('select company_id,descriptions from company_descriptions_fact_sheet')
des_df=pd.DataFrame(cur.fetchall())

In [54]:
des_df.to_pickle('/Users/pragalbh.devsingh/Downloads/all_factsheets.pkl')   

In [55]:
des_df

,company_id,descriptions
0,106,"subscription fees, pricing strategies, detaile..."
1,166,"shipping boxes, pricing strategies, marketing ..."
2,250,"pricing strategies, personalized financial, cr..."
3,367,"pricing strategies, styling, uae, nail care, m..."
4,521,"management consultancy, internet marketing ser..."
...,...,...
1037173,1460850,"non - binding, business premises, loan applica..."
1037174,1460871,"online competitive data, online campaigns, onl..."
1037175,1460916,"online searches, online advertising campaigns,..."
1037176,1460966,"leadership, website development, happiness, ta..."


In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('/Users/pragalbh.devsingh/Downloads/qexp_deployment/qexp_model_deployment/e5_production_model')


In [16]:
from emb_search import search

In [17]:
query='companies with target audience as Tobacco industry'
emb=model.encode(query,batch_size=100)

In [18]:
res=search(emb.tolist(),cur)

In [8]:
print(res.descriptions.iloc[1])

marketing strategies, sales training component, revenue model verification, revenue structure, marketing materials, content strategy development, coffee producer, content management, cookies, sales volumes, brand building, content marketing campaigns, brand awareness enhancement, revenue generation model, content creation, brand selling, revenue streams, marketing software, revenue model specifics, advertising platforms, revenue figures, revenue sharing, marketing purposes, revenue cycle management, magazine subscriptions, content company, marketing agencies, tea, marketing technologies, advertising revenue, marketing services, revenue breakdown, beverage companies, marketing agency, marketing automation platforms, brand recognition, service fees, revenue stream, 
 here 's the company fact sheet for the company hab , based on the information gathered from their website : 

 * * company name :* * the company hab 

 * * products , services , and offerings :* * the company hab is a coffee

In [9]:
import pandas as pd
aspect_queries=pd.read_pickle('/Users/pragalbh.devsingh/Downloads/aspect_queries.pkl')

In [20]:
aspect_queries.keys()

dict_keys(['Industry', 'Customer Segment', 'Products & Services', 'Business Model', 'Technology Used', 'Revenue Model'])

In [21]:
aspect_queries['Customer Industry']=aspect_queries['Industry']

In [25]:
aspect_query_transformation_f={'Industry':lambda x:f'Companies which are building solutions in the {x} industry',
                               'Customer Segment':lambda x:f'Companies with target audience as {x} customer segment',
                               'Business Model':lambda x:f'Companies with {x} business model',
                               'Technology Used':lambda x:f'Companies which are using the {x} technology to build their solutions',
                               'Revenue Model':lambda x:f'Companies which are using the {x} revenue model',
                               'Customer Industry':lambda x:f'Companies whose customer or Target Audience  is a part of  {x} industry',
                               'Products & Services':lambda x:f'Companies which are offering {x}'}


In [26]:
# aspect_query_fetached_positives={}
# for aspect,queries in aspect_queries.items():
#     queries=queries[1]['queries']
#     aspect_query_fetached_positives[aspect]={}
#     for query in queries:
#         transformed_query=aspect_query_transformation_f[aspect](query)
#         emb=model.encode(transformed_query,batch_size=100)
#         res=search(emb.tolist(),cur)
#         aspect_query_fetached_positives[aspect][query]=res

In [28]:
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

aspect_query_fetched_positives = {}
for aspect, queries in tqdm(aspect_queries.items()):
    queries = queries[1]['queries']
    aspect_query_fetched_positives[aspect] = {}
    
    # Transform all queries for the current aspect
    transformed_queries = [(aspect_query_transformation_f[aspect](query), query) for query in queries]
    
    # Batch encode all transformed queries at once
    query_texts = [tq[0] for tq in transformed_queries]
    embeddings = model.encode(query_texts, batch_size=100)
    
    # Process searches in parallel with ThreadPoolExecutor (better for I/O-bound tasks)
    with ThreadPoolExecutor(max_workers=min(32, len(queries))) as executor:
        # Define a worker function for parallel processing
        def search_worker(search_input):
            conn=s2.connect(host='linkup-singlestore.prod.synaptic.services',
                     port=3306,password='ManagerMemSQL@synaptic_production',user='manager_user',
                     database='labs_core_reloaded_production',connect_timeout=30,results_type='dict')
            cur=conn.cursor()
            emb, query = search_input
            res = search(emb, cur)
            return query, res
        
        # Create a list of (embedding, query) pairs
        search_inputs = [(emb.tolist(), query) for emb, (_, query) in zip(embeddings, transformed_queries)]
        
        # Execute parallel searches
        future_to_query = {executor.submit(search_worker, search_input): search_input[1]
                          for search_input in search_inputs}
        
        # Collect results as they complete
        for future in tqdm(concurrent.futures.as_completed(future_to_query)):
            query, res = future.result()
            aspect_query_fetched_positives[aspect][query] = res

99it [00:16,  6.02it/s]0:00<?, ?it/s]
102it [00:17,  5.95it/s]:17<01:42, 17.10s/it]
98it [00:16,  5.89it/s]0:34<01:27, 17.49s/it]
10it [00:04,  2.47it/s]0:52<01:09, 17.44s/it]
30it [00:07,  4.08it/s]0:56<00:37, 12.37s/it]
25it [00:04,  5.40it/s]1:04<00:21, 10.75s/it]
99it [00:16,  6.05it/s]1:09<00:08,  8.86s/it]
100%|██████████| 7/7 [01:26<00:00, 12.36s/it]


In [ ]:
for aspect,q_dict in aspect_query_fetched_positives.items():
    for q,res in q_dict.items():
        
        


In [29]:
import pickle
with open('/Users/pragalbh.devsingh/Downloads/aspect_query_fetched_positives.pkl','wb') as f:
    pickle.dump(aspect_query_fetched_positives,f)

In [45]:
aspect_query_fetched_positives=pd.read_pickle('/Users/pragalbh.devsingh/Downloads/aspect_query_fetched_positives.pkl')

In [30]:
aspect_query_fetched_positives['Customer Industry'].keys()

dict_keys(['Automotive', 'Mining', 'Retail', 'Banking', 'Food and Beverage', 'Finance', 'Entertainment', 'Fashion', 'Logistics', 'Agriculture', 'Transportation', 'Utilities', 'Insurance', 'Aerospace', 'Manufacturing', 'Telecommunications', 'Real Estate', 'Renewable Energy', 'Technology', 'Defense', 'Energy', 'Hospitality', 'Chemicals', 'Education', 'Healthcare', 'Construction', 'Consumer Electronics', 'Pharmaceutical', 'Beauty', 'Biotechnology', 'Legal', 'Media', 'Public Sector', 'Non-Profit', 'Sports', 'Textiles', 'Gaming', 'Furniture', 'Ceramics', 'Marine', 'Forestry', 'Information Technology', 'Cybersecurity', 'Jewelry', 'Tobacco', 'Fishing', 'Consulting', 'Plastics', 'Event Management', 'Broadcasting', 'Travel', 'Printing', 'Human Resources', 'Paper and Pulp', 'Music Industry', 'E-commerce', 'Tourism', 'Hardware Manufacturing', 'Petroleum', 'Metals', 'Publishing', 'Glass', 'Software Development', 'Advertising', 'Film Production', 'Animation', 'Virtual Reality', 'Augmented Reality',

In [33]:
print(aspect_query_fetched_positives['Customer Industry']['Automotive'].descriptions.iloc[0])

marketing teams, pricing strategies, marketing performance, motortrend, automotive business decisions, pricing specifics, marketing materials, automotive enthusiasts, automotive suppliers, advertising companies, data analysis, advertising photography, pricing information, marketing industry, marketing content, pricing structure, automotive brands, advertising agencies, marketing manager, revenue model specifics, marketing message, strategic marketing, marketing campaigns, automotive publications, marketing agencies, advertising revenue, pricing tiers, audience engagement, automotive media, 
 okay , here 's the the company fact sheet based on the information gathered from their website : 

 * * the company fact sheet * * 

 * * 1 .   company overview :* * 

 the company ( jam ) is a full - service automotive media partner .    they do n't appear to be a single entity but rather a division or brand under hearst autos , encompassing several well - known automotive publications .    this f

In [44]:
aspect_query_fetched_positives['Customer Industry']['Automotive']

,company_id,descriptions,emb_score
0,49975,"marketing teams, pricing strategies, marketing...",0
1,896891,"automotive parts manufacturing, digital learni...",0
2,1096840,"course, manpower services, channel, configurab...",0
3,469682,"automotive topics, automotive straps, affiliat...",0
4,1301033,"business strategies, online automotive media i...",0
...,...,...,...
95,511741,"ice vehicle market, investment strategy, inves...",0
96,1018901,"subscription option, subscription fees, pricin...",0
97,827377,"visual presentation, jdm, professional quality...",0
98,174687,"vehicle sales, seo optimization, community eng...",0


In [48]:
aspect_query_fetched_positives['Customer Industry']['Automotive'].dtypes


company_id        int64
descriptions     object
emb_score       float64
dtype: object

In [47]:
aspect_query_fetched_positives['Customer Industry']['Automotive'].emb_score=aspect_query_fetched_positives['Customer Industry']['Automotive'].emb_score.astype(float)
aspect_query_fetched_positives['Customer Industry']['Automotive'].company_id=aspect_query_fetched_positives['Customer Industry']['Automotive'].company_id.astype(int)

In [38]:
print(aspect_query_fetched_positives['Customer Industry']['Retail'].descriptions.iloc[3])

merchandise, demographics, web scraping, video games publishers, fosters, bull moose, community engagement, unified commerce, psychographics, program details, buybacks program, affiliate marketing, program management, competitive landscape analysis, program offerings, program content, curated selection, partnerships, 
 okay , here 's the final company fact sheet for bull moose , based on our web scraping and external research efforts .    keep in mind that due to the limited information publicly available on their website , some aspects require further investigation for a truly comprehensive analysis . 

 * * company fact sheet : bull moose * * 

 * * 1 .   overview :* * the company is a brick - and - mortar and e - commerce retailer specializing in the sale of physical media and merchandise .    their primary focus appears to be on serving a niche market of consumers who value physical products and the in - store experience . 

 * * 2 .   products , services , and offerings :* * 

 * 

In [10]:
aspect_query_fetached_positives={}
for aspect,queries in aspect_queries.items():
    queries=queries[1]['queries']
    aspect_query_fetached_positives[aspect]={}
    for query in queries:
        transformed_query=aspect_query_transformation_f[aspect](query)
        emb=model.encode(transformed_query,batch_size=100)
        res=search(emb.tolist(),cur)
        aspect_query_fetached_positives[aspect][query]=res
        


{'Industry': (True,
  {'queries': ['Healthcare',
    'Finance',
    'Retail',
    'Automotive',
    'Technology',
    'Education',
    'Manufacturing',
    'Construction',
    'Real Estate',
    'Energy',
    'Telecommunications',
    'Transportation',
    'Agriculture',
    'Pharmaceutical',
    'Food and Beverage',
    'Hospitality',
    'Entertainment',
    'Media',
    'Aerospace',
    'Defense',
    'Consumer Electronics',
    'Fashion',
    'Beauty',
    'Insurance',
    'Banking',
    'Legal',
    'Logistics',
    'Mining',
    'Chemicals',
    'Biotechnology',
    'Renewable Energy',
    'Utilities',
    'Public Sector',
    'Non-Profit',
    'Sports',
    'Gaming',
    'Textiles',
    'Furniture',
    'Petroleum',
    'Metals',
    'Paper and Pulp',
    'Printing',
    'Marine',
    'Fishing',
    'Forestry',
    'Plastics',
    'Ceramics',
    'Glass',
    'Jewelry',
    'Tobacco',
    'Advertising',
    'Consulting',
    'Event Management',
    'Travel',
    'Tourism',
    '

In [19]:
model.encode(,batch_size=100)

SyntaxError: invalid syntax (1753005094.py, line 1)